# EP06 — Generative AI: GANs & Diffusion Models
**Exam Relevance:** 2025 Q7.1 (9 marks) | 2024 Q6 (MCQ + questions)

Topics:
1. GAN architecture: Generator vs Discriminator objectives
2. Diffusion models: core idea of noising and denoising
3. Autoencoders and VAEs (2024 MCQ)
4. Why diffusion uses small steps, not one big jump

---

## 1. GAN — Training Objectives

**2025 Q7.1a asked:** *Briefly describe the training objectives of the generator and discriminator.*

### The Adversarial Game:

**Generator (G):**
- Takes random noise z as input
- Generates fake images G(z)
- Objective: **fool the discriminator** — make G(z) look so realistic that D predicts it as real (D(G(z)) → 1)
- Wants to MAXIMISE D(G(z)) (or equivalently minimise log(1 - D(G(z))))

**Discriminator (D):**
- Takes an image (real or fake) as input
- Objective: **correctly distinguish real from fake** 
  - D(real image) → 1 (output high probability for real)
  - D(G(z)) → 0 (output low probability for fake)
- Wants to MAXIMISE log(D(x)) + log(1 - D(G(z)))

**Together:** minimax game — G minimises, D maximises the same objective

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Mini GAN on 1D data — to show the training dynamics
# Real data: samples from N(4, 0.5)
# Generator: maps noise N(0,1) → tries to match real dist
# ============================================================

torch.manual_seed(42)

class Generator1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, z): return self.net(z)

class Discriminator1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

G = Generator1D()
D = Discriminator1D()
opt_G = torch.optim.Adam(G.parameters(), lr=0.001)
opt_D = torch.optim.Adam(D.parameters(), lr=0.001)
criterion = nn.BCELoss()

g_losses, d_losses = [], []
snapshots = {}

for epoch in range(2000):
    batch_size = 128
    
    # Real data from N(4, 0.5)
    real = torch.randn(batch_size, 1) * 0.5 + 4.0
    noise = torch.randn(batch_size, 1)
    fake = G(noise).detach()
    
    # --- Train Discriminator ---
    opt_D.zero_grad()
    d_real = criterion(D(real), torch.ones(batch_size, 1))
    d_fake = criterion(D(fake), torch.zeros(batch_size, 1))
    d_loss = d_real + d_fake
    d_loss.backward()
    opt_D.step()
    
    # --- Train Generator ---
    opt_G.zero_grad()
    noise2 = torch.randn(batch_size, 1)
    g_loss = criterion(D(G(noise2)), torch.ones(batch_size, 1))  # G wants D to output 1
    g_loss.backward()
    opt_G.step()
    
    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())
    
    if epoch in [0, 100, 500, 1999]:
        with torch.no_grad():
            snapshots[epoch] = G(torch.randn(1000, 1)).numpy().flatten()

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curves
axes[0].plot(g_losses, label='Generator loss', alpha=0.7, color='blue')
axes[0].plot(d_losses, label='Discriminator loss', alpha=0.7, color='red')
axes[0].set_title('GAN Training Losses')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution evolution
real_data = np.random.normal(4, 0.5, 1000)
axes[1].hist(real_data, bins=40, alpha=0.5, label='Real (target N(4,0.5))', color='green', density=True)
for epoch, label, alpha in [(0, 'epoch 0', 0.3), (100, 'epoch 100', 0.5), 
                              (500, 'epoch 500', 0.7), (1999, 'epoch 2000', 0.9)]:
    axes[1].hist(snapshots[epoch], bins=40, alpha=alpha, label=f'Generated {label}', density=True)
axes[1].set_title('Generator Distribution Over Training')
axes[1].set_xlabel('Value')
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

# GAN architecture diagram
axes[2].axis('off')
axes[2].set_xlim(0, 10)
axes[2].set_ylim(0, 6)

# Noise
axes[2].add_patch(plt.Rectangle((0, 2.5), 1.5, 1, color='lightgray', ec='black'))
axes[2].text(0.75, 3.0, 'Noise z', ha='center', va='center', fontsize=9)

# Generator
axes[2].add_patch(plt.Rectangle((2, 2.5), 2, 1, color='lightblue', ec='blue'))
axes[2].text(3.0, 3.0, 'Generator\nG(z)', ha='center', va='center', fontsize=9)
axes[2].annotate('', xy=(2, 3.0), xytext=(1.5, 3.0), arrowprops=dict(arrowstyle='->', color='black'))

# Fake image
axes[2].add_patch(plt.Rectangle((5, 2.5), 1.5, 1, color='#FFB3B3', ec='black'))
axes[2].text(5.75, 3.0, 'Fake', ha='center', va='center', fontsize=9)
axes[2].annotate('', xy=(5, 3.0), xytext=(4, 3.0), arrowprops=dict(arrowstyle='->', color='blue'))

# Real image
axes[2].add_patch(plt.Rectangle((5, 4.5), 1.5, 1, color='#B3FFB3', ec='black'))
axes[2].text(5.75, 5.0, 'Real', ha='center', va='center', fontsize=9)

# Discriminator
axes[2].add_patch(plt.Rectangle((7.5, 3.0), 2, 1.5, color='lightyellow', ec='red'))
axes[2].text(8.5, 3.75, 'Discriminator\nD', ha='center', va='center', fontsize=9)
axes[2].annotate('', xy=(7.5, 3.25), xytext=(6.5, 3.0), arrowprops=dict(arrowstyle='->', color='red'))
axes[2].annotate('', xy=(7.5, 4.0), xytext=(6.5, 5.0), arrowprops=dict(arrowstyle='->', color='red'))

axes[2].text(8.5, 2.5, 'Real/Fake?', ha='center', fontsize=8, color='red')
axes[2].set_title('GAN Architecture', fontsize=11)

plt.tight_layout()
plt.show()

## 2. Diffusion Models — Core Idea

**2025 Q7.1b asked:** *What is the core idea behind image generation using diffusion models?*

### Answer:
Diffusion models work in two phases:

1. **Forward process (noising):** Gradually add Gaussian noise to a real image over T timesteps until it becomes pure noise. This is fixed/not learned.

2. **Reverse process (denoising):** A neural network learns to reverse each small noise step — predicting the slightly less noisy image. By iterating T steps, we can start from pure noise and denoise it into a realistic image.

**2024 Q6.1(5) — Why small steps, not one big jump?**  
Answer: **B — Easier to learn and control the process**  
- Each small denoising step is a simpler, more tractable problem
- The network only needs to predict a small amount of noise removal
- One big jump would require predicting a sharp image directly from pure noise — much harder

In [ ]:
# Visualise the forward diffusion process (adding noise gradually)
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulate forward diffusion on a simple 1D signal (representing an image pixel value)
# Real value = 0.8 (a bright pixel)
x_0 = 0.8
T = 20  # timesteps
beta = np.linspace(0.01, 0.3, T)  # noise schedule

x_t_values = [x_0]
x_t = x_0
for t in range(T):
    noise = np.random.normal(0, 1)
    x_t = np.sqrt(1 - beta[t]) * x_t + np.sqrt(beta[t]) * noise
    x_t_values.append(x_t)

# Plot forward process
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(range(T+1), x_t_values, 'b-o', markersize=5)
axes[0].axhline(0, color='red', linestyle='--', label='Pure noise baseline (mean=0)')
axes[0].axhline(x_0, color='green', linestyle='--', label=f'Original value ({x_0})')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('Pixel value')
axes[0].set_title('Forward Diffusion Process\n(gradually adding noise)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Visualise noise levels on a fake image
steps_to_show = [0, 5, 10, 15, 19]
# Simulate a 16x16 "image" at different noise levels
x_img = np.ones((16, 16)) * 0.7 + np.random.randn(16, 16) * 0.05  # clean image

images_at_steps = []
x_curr = x_img.copy()
for t in range(T):
    noise = np.random.randn(*x_curr.shape)
    x_curr = np.sqrt(1 - beta[t]) * x_curr + np.sqrt(beta[t]) * noise
    if t in steps_to_show:
        images_at_steps.append((t, x_curr.copy()))

n_show = len(steps_to_show)
for i, (step, img) in enumerate(images_at_steps):
    axes[1].add_patch(plt.Rectangle((i*1.1 + 0.05, 0.05), 1, 1, color='white', ec='black'))
    # Show as mini heatmap
    ax2_inset = fig.add_axes([0.53 + i*0.085, 0.22, 0.075, 0.55])
    ax2_inset.imshow(img, cmap='gray', vmin=-1, vmax=1)
    ax2_inset.set_title(f't={step}', fontsize=8)
    ax2_inset.axis('off')

axes[1].set_xlim(-0.1, 5.7)
axes[1].set_ylim(-0.2, 1.3)
axes[1].axis('off')
axes[1].set_title('Forward Process: Clean → Noise\n(each panel = increasing noise added)', fontsize=10)

plt.tight_layout()
plt.show()

print("Forward process: x_0 (real image) → x_T (pure noise)  [NOT learned]")
print("Reverse process: x_T (pure noise) → x_0 (real image)  [LEARNED by neural network]")
print("Generation: sample x_T ~ N(0,I), then run reverse process T times")

## 3. Autoencoders and VAEs (2024 MCQ Review)

**2024 Q6.1 answers:**
1. AE main parts → **B: Encoder and Decoder**
2. KL divergence in VAE → **B: encourages learned distribution to be close to a prior (normal)**
3. GAN main parts → **A: Generator and Discriminator**
4. Discriminator aim → **B: Tell real data apart from fake data**
5. Why small denoising steps → **B: Easier to learn and control the process**

In [ ]:
# Visualise AE vs VAE latent space conceptually
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AE — discrete points in latent space
np.random.seed(42)
ae_clusters = [
    np.random.randn(30, 2) * 0.2 + [2, 2],
    np.random.randn(30, 2) * 0.2 + [-2, 2],
    np.random.randn(30, 2) * 0.2 + [0, -2],
]
for i, cluster in enumerate(ae_clusters):
    axes[0].scatter(cluster[:,0], cluster[:,1], label=f'Class {i}', s=20)
axes[0].set_title('Autoencoder Latent Space\n(clusters may have gaps — sampling unreliable)')
axes[0].set_xlabel('z₁')
axes[0].set_ylabel('z₂')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VAE — smooth latent space from KL regularization
vae_clusters = [
    np.random.randn(30, 2) * 0.8 + [1, 1],
    np.random.randn(30, 2) * 0.8 + [-1, 1],
    np.random.randn(30, 2) * 0.8 + [0, -1],
]
for i, cluster in enumerate(vae_clusters):
    axes[1].scatter(cluster[:,0], cluster[:,1], label=f'Class {i}', s=20)

# Show the N(0,1) prior
theta = np.linspace(0, 2*np.pi, 100)
axes[1].plot(np.cos(theta)*2, np.sin(theta)*2, 'k--', alpha=0.4, label='N(0,1) prior (2σ)')
axes[1].set_title('VAE Latent Space\n(KL loss forces clusters toward N(0,1) → smooth space)')
axes[1].set_xlabel('z₁')
axes[1].set_ylabel('z₂')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("VAE key insight: KL term forces the latent distribution toward N(0,1).")
print("This means you can sample any z ~ N(0,1) and decode it into a valid image.")
print("In an AE, random z might fall in a 'gap' between clusters → garbage output.")

## Exam Quick-Reference Summary

| Model | Two parts | Training objective |
|---|---|---|
| AE | Encoder + Decoder | Reconstruct input |
| VAE | Encoder + Decoder | Reconstruct + KL divergence (smooth latent space) |
| GAN | Generator + Discriminator | G fools D; D distinguishes real vs fake |
| Diffusion | U-Net denoiser | Learn to remove small amounts of noise at each step |

**GAN objectives (1 sentence each):**
- **Generator:** minimise the probability that the discriminator correctly identifies its output as fake
- **Discriminator:** maximise accuracy at classifying real images as real and fake images as fake

**Diffusion core idea (2 sentences):**
> Images are gradually corrupted with noise over T steps (forward process). A neural network learns to reverse this process step by step, enabling generation by starting from pure noise and iteratively denoising.

**Why small steps?** Each step is a simpler regression problem; one big step from noise to image is too hard to learn.